In [1]:
#! Import dependencies and mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import hashlib
import json
import math
import os
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type != 'cuda':
    raise RuntimeError('Select Runtime > Change runtime type > T4 GPU before training Task 4C.')

print('PyTorch:', torch.__version__, '| device:', torch.cuda.get_device_name(0))

Mounted at /content/drive
PyTorch: 2.11.0+cu128 | device: Tesla T4


## Cell 2 — Paths, dependency audit, and controlled configuration

Require Task 4A's immutable split, random initialization and shared model code. GCSC receives **separate** `gcsc/` checkpoints, results and caches; the only intentional recipe difference is RandAugment.

In [2]:
#! Reuse Vanilla hyperparameters with only the specified augmentation change
ROOT = Path('/content/drive/MyDrive/pa1-beyond-iid')
TASK = ROOT / 'task4'
CODE_DIR = TASK / 'code'
DATA_DIR = TASK / 'data' / 'cifar10'
SPLIT_DIR = TASK / 'splits'
CONFIG_DIR = TASK / 'configs'
CKPT_DIR = TASK / 'checkpoints' / 'gcsc'
RESULT_DIR = TASK / 'results' / 'gcsc'
CACHE_DIR = TASK / 'cache' / 'gcsc'

SEED, EPOCHS, BATCH_SIZE, NUM_WORKERS = 6304, 100, 128, 2
LR, MOMENTUM, WEIGHT_DECAY = 0.1, 0.9, 5e-4
SPLIT_FILE = SPLIT_DIR / 'cifar10_stratified_90_10_seed6304.npz'
INITIAL_PATH = TASK / 'checkpoints' / 'shared_random_resnet18_seed6304.pt'
CORE_FILE = CODE_DIR / 'cifar_osr_core_v1.py'
VANILLA_CONFIG_PATH = CONFIG_DIR / 'vanilla.json'
CONFIG_FILE = CONFIG_DIR / 'gcsc.json'
BEST_PATH = CKPT_DIR / 'best.pt'
LAST_PATH = CKPT_DIR / 'last.pt'
HISTORY_FILE = RESULT_DIR / 'history.csv'

required = [SPLIT_FILE, INITIAL_PATH, CORE_FILE, VANILLA_CONFIG_PATH]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(
        'Run Task 4A setup through Cell 5 before Task 4C. Missing:\n' + '\n'.join(missing)
    )

for directory in [CKPT_DIR, RESULT_DIR, CACHE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

VANILLA_CONFIG = json.loads(VANILLA_CONFIG_PATH.read_text())
CORE_SHA = hashlib.sha256(CORE_FILE.read_bytes()).hexdigest()
VANILLA_CONFIG_SHA = hashlib.sha256(json.dumps(VANILLA_CONFIG, sort_keys=True).encode()).hexdigest()
CONFIG = dict(VANILLA_CONFIG)
CONFIG['experiment'] = 'task4_gcsc'
CONFIG['augmentation'] = ('RandomCrop(32,padding=4), RandomHorizontalFlip(), '
                          'RandAugment(num_ops=2,magnitude=9)')
CONFIG['shared_core_sha256'] = CORE_SHA

expected = {
    'seed': SEED, 'epochs': EPOCHS, 'batch_size': BATCH_SIZE,
    'learning_rate': LR, 'momentum': MOMENTUM, 'weight_decay': WEIGHT_DECAY,
    'selection': 'highest CIFAR-10 validation accuracy', 'precision': 'float32',
    'shared_code_version': 'cifar_osr_core_v1',
}
if any(VANILLA_CONFIG.get(key) != value for key, value in expected.items()):
    raise RuntimeError('Vanilla config does not match the agreed controlled experiment settings.')
if VANILLA_CONFIG.get('augmentation') != 'RandomCrop(32,padding=4), RandomHorizontalFlip()':
    raise RuntimeError('Expected the original Vanilla crop+flip recipe for controlled comparison.')

CONFIG_SHA = hashlib.sha256(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()
if CONFIG_FILE.exists():
    if json.loads(CONFIG_FILE.read_text()) != CONFIG:
        raise RuntimeError('Existing GCSC config differs; refusing an incompatible rerun.')
else:
    CONFIG_FILE.write_text(json.dumps(CONFIG, indent=2) + '\n')

SPLIT_SHA = hashlib.sha256(SPLIT_FILE.read_bytes()).hexdigest()
INITIAL_SHA = hashlib.sha256(INITIAL_PATH.read_bytes()).hexdigest()
print('Independent GCSC checkpoints:', CKPT_DIR)
print('Shared initial weights SHA:', INITIAL_SHA)
print('GCSC config SHA:', CONFIG_SHA)


Independent GCSC checkpoints: /content/drive/MyDrive/pa1-beyond-iid/task4/checkpoints/gcsc
Shared initial weights SHA: badc9207a8c7ca962fbc1b7ba972e0652cd2b3b406342b70be7644a0592ab590
GCSC config SHA: d5425a1abfef26a2707f7ea41f12ac7ac2403939e1e1793d2ebbb174f8fb0b2a


## Cell 3 — Reuse the shared ResNet and audit the augmentation

Import Task 4A's *existing* `cifar_osr_core_v1.py` rather than writing it again. Check the exact crop → flip → RandAugment → ToTensor → Normalize order and test a synthetic RGB image.

In [3]:
#! Import the EXISTING shared module; do not regenerate or edit its Python file
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
import cifar_osr_core_v1 as shared
if Path(shared.__file__).resolve() != CORE_FILE.resolve():
    raise RuntimeError('A different shared module was imported from ' + str(shared.__file__))

seed_all = shared.seed_all
make_resnet18 = shared.make_resnet18
forward_features_logits = shared.forward_features_logits
training_transform = shared.training_transform
evaluation_transform = shared.evaluation_transform
atomic_torch_save = shared.atomic_torch_save
extract_known_outputs = shared.extract_known_outputs

#! Confirm the *only* augmentation addition is exactly 2-op magnitude-9 RandAugment
ordinary = training_transform(False).transforms
strong = training_transform(True).transforms
ordinary_names = [type(op).__name__ for op in ordinary]
strong_names = [type(op).__name__ for op in strong]
if ordinary_names != ['RandomCrop', 'RandomHorizontalFlip', 'ToTensor', 'Normalize']:
    raise RuntimeError('Vanilla transform order changed: ' + str(ordinary_names))
if strong_names != ['RandomCrop', 'RandomHorizontalFlip', 'RandAugment', 'ToTensor', 'Normalize']:
    raise RuntimeError('GCSC transform order changed: ' + str(strong_names))
if (strong[2].num_ops != 2 or strong[2].magnitude != 9
        or strong[0].padding != 4 or strong[0].size != (32, 32)):
    raise RuntimeError('RandAugment/crop parameters do not match assignment.')

from PIL import Image
synthetic_rgb = Image.new('RGB', (32, 32), color=(84, 121, 202))
probe = training_transform(True)(synthetic_rgb)
if probe.shape != (3, 32, 32) or not bool(torch.isfinite(probe).all()):
    raise RuntimeError('RandAugment + normalization synthetic test failed.')
print('Shared code reused WITHOUT modification; augmentation order/parameters: PASS')


Shared code reused WITHOUT modification; augmentation order/parameters: PASS


## Cell 4 — Reuse and audit the exact CIFAR-10 split

Only the official CIFAR-10 **training** partition is accessed. Reusing the existing split prevents an accidental different validation set; neither CIFAR-10 test nor CIFAR-100 is accessed.

In [4]:
#! Read and independently validate the EXISTING 45k/5k stratified split
raw_train = datasets.CIFAR10(root=str(DATA_DIR), train=True, download=False)
targets = np.asarray(raw_train.targets, dtype=np.int64)
with np.load(SPLIT_FILE, allow_pickle=False) as stored_split:
    train_idx = stored_split['train_idx'].astype(np.int64)
    val_idx = stored_split['val_idx'].astype(np.int64)

if (len(targets) != 50000 or len(train_idx) != 45000 or len(val_idx) != 5000
        or np.any(train_idx < 0) or np.any(val_idx < 0)
        or np.any(train_idx >= 50000) or np.any(val_idx >= 50000)
        or len(np.unique(np.r_[train_idx, val_idx])) != 50000
        or not np.array_equal(np.bincount(targets[train_idx], minlength=10), np.full(10, 4500))
        or not np.array_equal(np.bincount(targets[val_idx], minlength=10), np.full(10, 500))):
    raise RuntimeError('Shared train/validation split is not the prescribed 90/10 stratified partition.')
print('TRAIN:', len(train_idx), '| VALIDATION:', len(val_idx))
print('Same split SHA as Task 4A:', SPLIT_SHA)
del raw_train


TRAIN: 45000 | VALIDATION: 5000
Same split SHA as Task 4A: 58e3e0a58ca34ebab457ec178874817e597302512f91228ec13a69420b6164d8


## Cell 5 — Augmented loaders, identical initial weights and optimizer

Train on the original 45,000 CIFAR-10 training examples with RandAugment added. Reload *the exact Task 4A initial model weights*, not the trained Vanilla checkpoint. Maintain SGD 0.1/0.9/5e−4, cosine T_max=100 and batch 128.

In [5]:
#! Use the same data, SGD and cosine schedule as Vanilla; only augmentation changes
train_dataset = Subset(
    datasets.CIFAR10(root=str(DATA_DIR), train=True, download=False,
                     transform=training_transform(True)), train_idx.tolist()
)
train_clean_dataset = Subset(
    datasets.CIFAR10(root=str(DATA_DIR), train=True, download=False,
                     transform=evaluation_transform()), train_idx.tolist()
)
val_dataset = Subset(
    datasets.CIFAR10(root=str(DATA_DIR), train=True, download=False,
                     transform=evaluation_transform()), val_idx.tolist()
)

train_generator = torch.Generator()
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          drop_last=False, num_workers=NUM_WORKERS,
                          pin_memory=True, generator=train_generator)
train_clean_loader = DataLoader(train_clean_dataset, batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

seed_all(SEED)
model = make_resnet18(10)
initial = torch.load(INITIAL_PATH, map_location='cpu', weights_only=False)
if initial.get('seed') != SEED or initial.get('architecture') != VANILLA_CONFIG['architecture']:
    raise RuntimeError('Task 4A initial checkpoint is not compatible with GCSC.')
model.load_state_dict(initial['model_state_dict'], strict=True)
model = model.to(DEVICE)

optimizer = torch.optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM,
                            weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=0.)
criterion = torch.nn.CrossEntropyLoss()
print('Reused EXACT Vanilla random initialization; parameters:',
      sum(p.numel() for p in model.parameters()))
print('GCSC augmented batches:', len(train_loader), '| same 45,000 training examples per epoch')


Reused EXACT Vanilla random initialization; parameters: 11173962
GCSC augmented batches: 352 | same 45,000 training examples per epoch


## Cell 6 — Independent architecture and gradient checks

A synthetic batch confirms the CIFAR stem and 512-D features, matches torchvision logits and verifies finite float32 gradients **without taking an optimizer step**.

In [6]:
#! Sanity-check feature extraction and one float32 backward without updating weights
model.eval()
synthetic = torch.randn(2, 3, 32, 32, device=DEVICE)
with torch.no_grad():
    f, z = forward_features_logits(model, synthetic)
    official = model(synthetic)
if f.shape != (2, 512) or z.shape != (2, 10) or not torch.allclose(z, official, atol=1e-5):
    raise RuntimeError('CIFAR ResNet architecture/feature extraction test failed.')
model.zero_grad(set_to_none=True)
_, z = forward_features_logits(model, synthetic)
probe_loss = F.cross_entropy(z, torch.tensor([0, 1], device=DEVICE))
probe_loss.backward()
if (not bool(torch.isfinite(probe_loss))
        or model.fc.weight.grad is None
        or not bool(torch.isfinite(model.fc.weight.grad).all())):
    raise RuntimeError('Synthetic float32 forward/backward produced non-finite values.')
model.zero_grad(set_to_none=True)
del synthetic, f, z, official, probe_loss
print('GCSC model architecture, 512-D features and FP32 gradients: PASS')

GCSC model architecture, 512-D features and FP32 gradients: PASS


## Cell 7 — Safe resumption and validation

Load a finished-epoch GCSC checkpoint only. An interrupted epoch is discarded and restored from the previous complete epoch; an interrupted update of `best.pt` can be reconstructed from the committed `last.pt`. Check model, split, code and config identities before resuming.

In [7]:
#! Restore only fully committed GCSC epochs; repair an interrupted best-file write
history = []
best_acc, best_epoch, start_epoch = -1.0, 0, 1

def best_payload(epoch, accuracy, state_dict):
    return {
        'method': 'gcsc', 'epoch': epoch, 'seed': SEED, 'best_val_accuracy': accuracy,
        'config_sha256': CONFIG_SHA, 'split_sha256': SPLIT_SHA,
        'initial_sha256': INITIAL_SHA, 'shared_core_sha256': CORE_SHA,
        'vanilla_config_sha256': VANILLA_CONFIG_SHA,
        'model_state_dict': {k: v.detach().cpu().clone() for k, v in state_dict.items()},
    }

def validate_saved_provenance(saved, label):
    for key, expected in [
        ('config_sha256', CONFIG_SHA), ('split_sha256', SPLIT_SHA),
        ('initial_sha256', INITIAL_SHA), ('shared_core_sha256', CORE_SHA),
        ('vanilla_config_sha256', VANILLA_CONFIG_SHA),
    ]:
        if saved.get(key) != expected:
            raise RuntimeError(f'{label}: incompatible {key}. Do not resume different settings.')

if LAST_PATH.exists():
    saved = torch.load(LAST_PATH, map_location='cpu', weights_only=False)
    validate_saved_provenance(saved, 'GCSC last checkpoint')
    if (saved.get('method') != 'gcsc' or len(saved['history']) != saved['epoch']
            or saved['best_epoch'] < 1 or saved['best_epoch'] > saved['epoch']):
        raise RuntimeError('GCSC epoch history is inconsistent.')

    #! If interrupted after saving last but before new best, rebuild best from last.
    if saved['best_epoch'] == saved['epoch']:
        needs_repair = not BEST_PATH.exists()
        if not needs_repair:
            previous_best = torch.load(BEST_PATH, map_location='cpu', weights_only=False)
            needs_repair = previous_best.get('epoch', -1) < saved['best_epoch']
        if needs_repair:
            atomic_torch_save(best_payload(saved['epoch'], saved['best_val_accuracy'],
                                           saved['model_state_dict']), BEST_PATH)
            print('Recovered best checkpoint from fully committed last epoch.')

    if not BEST_PATH.exists():
        raise RuntimeError('Best GCSC checkpoint missing; do not assume training complete.')
    selected = torch.load(BEST_PATH, map_location='cpu', weights_only=False)
    validate_saved_provenance(selected, 'GCSC best checkpoint')
    if (selected.get('epoch') != saved['best_epoch']
            or selected.get('best_val_accuracy') != saved['best_val_accuracy']):
        raise RuntimeError('GCSC best/last disagree; inspect before resuming.')

    model.load_state_dict(saved['model_state_dict'], strict=True)
    optimizer.load_state_dict(saved['optimizer_state_dict'])
    scheduler.load_state_dict(saved['scheduler_state_dict'])
    history = list(saved['history'])
    best_acc, best_epoch = float(saved['best_val_accuracy']), int(saved['best_epoch'])
    start_epoch = int(saved['epoch']) + 1
elif BEST_PATH.exists():
    raise RuntimeError('Orphan GCSC best checkpoint without last; inspect before running.')
else:
    print('Starting GCSC from the original random weights, epoch 1.')

@torch.inference_mode()
def validation_metrics(net, loader):
    net.eval()
    total_loss, correct, count = 0.0, 0, 0
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        logits = net(images)
        loss = criterion(logits, labels)
        if not bool(torch.isfinite(loss).item()):
            raise FloatingPointError('Non-finite GCSC validation loss; no checkpoint selected.')
        total_loss += float(loss) * len(labels)
        correct += int((logits.argmax(dim=1) == labels).sum().item())
        count += len(labels)
    if count != 5000:
        raise RuntimeError('GCSC validation must process exactly 5,000 known examples.')
    return {'loss': total_loss / count, 'accuracy': correct / count}

print('Ready at complete epoch', start_epoch - 1, '| best epoch:', best_epoch)


Starting GCSC from the original random weights, epoch 1.
Ready at complete epoch 0 | best epoch: 0


## Cell 8 — Train all 100 epochs (no early stopping)

Per epoch: seed deterministically, run exactly 45,000 augmented known images through cross-entropy, validate on the fixed 5,000 clean known images, select the highest validation accuracy, and save full-epoch training state plus the best model. No FP16 or unknown examples.

In [8]:
#! Restart from the LAST completed epoch, never from a partially executed batch
if LAST_PATH.exists():
    rollback = torch.load(LAST_PATH, map_location='cpu', weights_only=False)
    validate_saved_provenance(rollback, 'Pre-training rollback')
    model.load_state_dict(rollback['model_state_dict'], strict=True)
    optimizer.load_state_dict(rollback['optimizer_state_dict'])
    scheduler.load_state_dict(rollback['scheduler_state_dict'])
    history = list(rollback['history'])
    best_acc, best_epoch = float(rollback['best_val_accuracy']), int(rollback['best_epoch'])
    start_epoch = int(rollback['epoch']) + 1
else:
    model.load_state_dict(initial['model_state_dict'], strict=True)
    optimizer = torch.optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM,
                                weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=0.)
    history, best_acc, best_epoch, start_epoch = [], -1.0, 0, 1

if start_epoch > EPOCHS:
    print('GCSC training already completed all 100 epochs; no redundant training.')
else:
    for epoch in range(start_epoch, EPOCHS + 1):
        started = time.perf_counter()
        seed_all(SEED + epoch)
        train_generator.manual_seed(SEED + 1009 * epoch)
        model.train()
        epoch_lr = optimizer.param_groups[0]['lr']
        total_loss, total_correct, total_count = 0.0, 0, 0

        for step, (images, labels) in enumerate(train_loader):
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            if not bool(torch.isfinite(loss).item()):
                raise FloatingPointError(f'GCSC loss non-finite at epoch={epoch}, batch={step}')
            loss.backward()
            if step % 100 == 0 and any(
                    p.grad is not None and not bool(torch.isfinite(p.grad).all().item())
                    for p in model.parameters()):
                raise FloatingPointError(f'GCSC gradient non-finite at epoch={epoch}, batch={step}')
            optimizer.step()
            count = len(labels)
            total_count += count
            total_loss += float(loss.detach()) * count
            total_correct += int((logits.detach().argmax(dim=1) == labels).sum().item())

        if total_count != 45000:
            raise RuntimeError('GCSC epoch must process all 45,000 training examples.')
        val = validation_metrics(model, val_loader)
        improved = val['accuracy'] > best_acc
        if improved:
            best_acc, best_epoch = float(val['accuracy']), epoch

        history.append({
            'epoch': epoch, 'learning_rate': epoch_lr,
            'train_loss': total_loss / total_count,
            'train_accuracy': total_correct / total_count,
            'val_loss': val['loss'], 'val_accuracy': val['accuracy'],
            'best_val_accuracy': best_acc, 'best_epoch': best_epoch,
            'elapsed_seconds': time.perf_counter() - started,
        })
        scheduler.step()
        #! Commit last FIRST, then best; the next run repairs a missing new best.
        atomic_torch_save({
            'method': 'gcsc', 'epoch': epoch, 'finished': epoch == EPOCHS,
            'model_state_dict': {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'history': history, 'best_val_accuracy': best_acc, 'best_epoch': best_epoch,
            'config_sha256': CONFIG_SHA, 'split_sha256': SPLIT_SHA,
            'initial_sha256': INITIAL_SHA, 'shared_core_sha256': CORE_SHA,
            'vanilla_config_sha256': VANILLA_CONFIG_SHA,
        }, LAST_PATH)
        if improved:
            atomic_torch_save(best_payload(epoch, best_acc, model.state_dict()), BEST_PATH)
        pd.DataFrame(history).to_csv(HISTORY_FILE, index=False)
        print(f'Epoch {epoch:03d}/{EPOCHS} | LR={epoch_lr:.6f} | '
              f'train loss={history[-1]["train_loss"]:.4f} | '
              f'train acc={history[-1]["train_accuracy"]:.4f} | '
              f'val acc={val["accuracy"]:.4f} | best={best_acc:.4f} (epoch {best_epoch}) | '
              f'{history[-1]["elapsed_seconds"]:.1f}s')

print('Completed GCSC epochs:', len(history), '| selected epoch:', best_epoch)


Epoch 001/100 | LR=0.100000 | train loss=2.2632 | train acc=0.1714 | val acc=0.2882 | best=0.2882 (epoch 1) | 42.9s
Epoch 002/100 | LR=0.099975 | train loss=1.8098 | train acc=0.3149 | val acc=0.4106 | best=0.4106 (epoch 2) | 44.4s
Epoch 003/100 | LR=0.099901 | train loss=1.5520 | train acc=0.4282 | val acc=0.4338 | best=0.4338 (epoch 3) | 47.9s
Epoch 004/100 | LR=0.099778 | train loss=1.3444 | train acc=0.5181 | val acc=0.5828 | best=0.5828 (epoch 4) | 46.6s
Epoch 005/100 | LR=0.099606 | train loss=1.1545 | train acc=0.5908 | val acc=0.6106 | best=0.6106 (epoch 5) | 48.0s
Epoch 006/100 | LR=0.099384 | train loss=1.0106 | train acc=0.6447 | val acc=0.7366 | best=0.7366 (epoch 6) | 47.3s
Epoch 007/100 | LR=0.099114 | train loss=0.8642 | train acc=0.6990 | val acc=0.7214 | best=0.7366 (epoch 6) | 46.9s
Epoch 008/100 | LR=0.098796 | train loss=0.7873 | train acc=0.7241 | val acc=0.7408 | best=0.7408 (epoch 8) | 46.4s
Epoch 009/100 | LR=0.098429 | train loss=0.7367 | train acc=0.7451 | val

## Cell 9 — Frozen best-model features, logits and alignment

Run **only after all 100 epochs**. Save unaugmented train and validation embeddings/logits in separate GCSC caches, indexed exactly like Task 4A. These are useful for later controlled analysis; no test or unknown images are seen.

In [9]:
#! Require the FULL 100-epoch run and selected model before saving frozen outputs
if not (LAST_PATH.is_file() and BEST_PATH.is_file()):
    raise RuntimeError('Finish GCSC training before extracting features.')
last_meta = torch.load(LAST_PATH, map_location='cpu', weights_only=False)
best_meta = torch.load(BEST_PATH, map_location='cpu', weights_only=False)
validate_saved_provenance(last_meta, 'Final last')
validate_saved_provenance(best_meta, 'Final best')
if (last_meta['epoch'] != EPOCHS or not last_meta['finished']
        or len(last_meta['history']) != EPOCHS
        or best_meta['method'] != 'gcsc'
        or best_meta['epoch'] != last_meta['best_epoch']
        or best_meta['best_val_accuracy'] != last_meta['best_val_accuracy']):
    raise RuntimeError('Do not extract outputs until all 100 GCSC epochs have finished.')

BEST_SHA = hashlib.sha256(BEST_PATH.read_bytes()).hexdigest()
model.load_state_dict(best_meta['model_state_dict'], strict=True)
model.eval()

for subset_name, indices, loader in [
    ('known_train', train_idx, train_clean_loader),
    ('known_val', val_idx, val_loader),
]:
    path = CACHE_DIR / f'{subset_name}_features_logits.npz'
    if path.exists():
        with np.load(path, allow_pickle=False) as existing:
            if (str(existing['best_checkpoint_sha256'].item()) != BEST_SHA
                    or not np.array_equal(existing['indices'], indices)
                    or not np.array_equal(existing['labels'], targets[indices])
                    or existing['features'].shape != (len(indices), 512)
                    or existing['logits'].shape != (len(indices), 10)):
                raise RuntimeError('Stale or incompatible GCSC feature cache: ' + str(path))
        print('Reusing saved:', path.name)
        continue

    features, logits, labels = extract_known_outputs(model, loader, DEVICE)
    if (features.shape != (len(indices), 512) or logits.shape != (len(indices), 10)
            or not np.array_equal(labels, targets[indices])
            or not np.isfinite(features).all() or not np.isfinite(logits).all()):
        raise RuntimeError('GCSC frozen feature cache failed shape, order, or finite checks.')
    temporary = path.with_name(path.name + '.tmp')
    with temporary.open('wb') as handle:
        np.savez_compressed(handle, features=features, logits=logits,
                            labels=labels, indices=indices,
                            best_checkpoint_sha256=np.array(BEST_SHA))
    os.replace(temporary, path)
    print('Saved:', subset_name, features.shape, logits.shape)

print('Only known TRAIN/VALIDATION features cached. No test or unknown data accessed.')


Saved: known_train (45000, 512) (45000, 10)
Saved: known_val (5000, 512) (5000, 10)
Only known TRAIN/VALIDATION features cached. No test or unknown data accessed.


## Cell 10 — Calibration, figures, and final verification

Use **only** the 5,000 known validation logits to define MLS unknownness as `−max(logits)` and freeze a 95th-percentile threshold (the same calibration principle as Task 4B). Save threshold provenance, validation scores and a training curve. The separate final Task 4 evaluation will access CIFAR-10 test and the fixed CIFAR-100 unknown subsets **only after every model and score is fixed**.

In [10]:
#! Fix MLS score/threshold using known validation only and audit all artifacts
with np.load(CACHE_DIR / 'known_val_features_logits.npz', allow_pickle=False) as cached:
    val_indices = cached['indices'].copy()
    val_labels = cached['labels'].copy()
    val_features = cached['features'].copy()
    val_logits = cached['logits'].copy()
    recorded_sha = str(cached['best_checkpoint_sha256'].item())
if (recorded_sha != BEST_SHA or not np.array_equal(val_indices, val_idx)
        or not np.array_equal(val_labels, targets[val_idx])
        or val_features.shape != (5000, 512) or val_logits.shape != (5000, 10)):
    raise RuntimeError('Frozen GCSC validation outputs are not aligned to the selected checkpoint.')

#! MLS unknownness: larger -max(logit) means more novel.
mls_unknownness = -np.max(val_logits.astype(np.float64), axis=1)
threshold = float(np.quantile(mls_unknownness, 0.95, method='linear'))
acceptance = float(np.mean(mls_unknownness <= threshold))
if not (np.isfinite(mls_unknownness).all() and 0.95 - 1/5000 <= acceptance <= 1):
    raise RuntimeError('GCSC MLS calibration failed known-validation verification.')

threshold_file = RESULT_DIR / 'mls_threshold_95pct_known_validation.json'
threshold_data = {
    'model': 'gcsc', 'best_checkpoint_sha256': BEST_SHA,
    'config_sha256': CONFIG_SHA, 'split_sha256': SPLIT_SHA,
    'score': 'MLS', 'unknownness_definition': '-max(logits)',
    'percentile': 0.95, 'quantile_method': 'linear',
    'accept_if': 'score <= threshold',
    'threshold': threshold, 'known_validation_acceptance': acceptance,
    'num_known_validation': 5000,
    'unknown_test_access': 'none',
}
if threshold_file.exists():
    if json.loads(threshold_file.read_text()) != threshold_data:
        raise RuntimeError('Existing GCSC threshold belongs to a different checkpoint/settings.')
else:
    tmp = threshold_file.with_name(threshold_file.name + '.tmp')
    tmp.write_text(json.dumps(threshold_data, indent=2, sort_keys=True) + '\n')
    os.replace(tmp, threshold_file)

score_file = RESULT_DIR / 'known_validation_mls.npz'
if score_file.exists():
    with np.load(score_file, allow_pickle=False) as old:
        if (str(old['best_checkpoint_sha256'].item()) != BEST_SHA
                or not np.array_equal(old['indices'], val_idx)
                or not np.array_equal(old['unknownness'], mls_unknownness)):
            raise RuntimeError('Existing GCSC validation MLS file differs; inspect before overwrite.')
else:
    tmp = score_file.with_name(score_file.name + '.tmp')
    with tmp.open('wb') as handle:
        np.savez_compressed(handle, indices=val_idx, labels=val_labels,
                            unknownness=mls_unknownness,
                            best_checkpoint_sha256=np.array(BEST_SHA))
    os.replace(tmp, score_file)

#! Save a training curve without changing model weights or checkpoint selection.
import matplotlib.pyplot as plt
frame = pd.DataFrame(last_meta['history'])
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(frame['epoch'], frame['train_loss'], label='Train CE')
axes[0].plot(frame['epoch'], frame['val_loss'], label='Val CE')
axes[0].set(xlabel='Epoch', ylabel='CE loss', title='GCSC loss')
axes[0].legend()
axes[1].plot(frame['epoch'], frame['train_accuracy'], label='Train accuracy')
axes[1].plot(frame['epoch'], frame['val_accuracy'], label='Val accuracy')
axes[1].axvline(best_meta['epoch'], linestyle='--', label='Selected epoch')
axes[1].set(xlabel='Epoch', ylabel='Accuracy', title='GCSC accuracy')
axes[1].legend()
fig.tight_layout()
fig.savefig(RESULT_DIR / 'training_curves.png', dpi=170)
plt.close(fig)

if not HISTORY_FILE.is_file() or len(pd.read_csv(HISTORY_FILE)) != EPOCHS:
    raise RuntimeError('GCSC history file must contain exactly 100 epochs.')
if len(last_meta['history']) != EPOCHS or best_meta['epoch'] != best_epoch:
    raise RuntimeError('Training history/selection failed final audit.')
print('TASK 4C COMPLETE')
print('Best GCSC validation accuracy:', best_meta['best_val_accuracy'],
      '| selected epoch:', best_meta['epoch'])
print('MLS threshold (95% known validation acceptance):', threshold,
      '| observed known validation acceptance:', acceptance)
print('Best checkpoint:', BEST_PATH)
print('Frozen known caches:', CACHE_DIR)
print('CIFAR-10 test and CIFAR-100 have NOT been accessed.')


TASK 4C COMPLETE
Best GCSC validation accuracy: 0.9532 | selected epoch: 89
MLS threshold (95% known validation acceptance): -5.990423154830933 | observed known validation acceptance: 0.95
Best checkpoint: /content/drive/MyDrive/pa1-beyond-iid/task4/checkpoints/gcsc/best.pt
Frozen known caches: /content/drive/MyDrive/pa1-beyond-iid/task4/cache/gcsc
CIFAR-10 test and CIFAR-100 have NOT been accessed.
